# Phase 3: Data Preparation

**CRISP-DM Phase Description:**  
This phase covers all activities to construct the final dataset from the initial raw data. Data preparation tasks are likely to be performed multiple times, and not in any prescribed order. This is typically the longest and most time-consuming phase of the CRISP-DM lifecycle.

---

In [1]:
# Standard library imports for this phase
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src import (
    AIRLINES_PATH,
    AIRPORTS_PATH,
    FLIGHTS_PATH,
    PREPARED_DATA_PATH,
    TARGET_COLUMN,
    add_phase3_features,
    cap_outliers_iqr,
    clean_phase3_missing_values,
    drop_duplicate_rows,
    filter_phase3_rows,
    load_airlines,
    load_airports,
    load_flights,
    merge_airline_lookup,
    merge_airport_lookup,
    save_prepared_dataset,
    select_phase3_data,
)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
%matplotlib inline

In [2]:
# Load the dataset from Phase 2 (update the path as needed)
DATA_PATH = FLIGHTS_PATH

df = load_flights(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Loaded dataset: 5819079 rows x 31 columns


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,2354.0,-11.0,21.0,15.0,205.0,194.0,169.0,1448,404.0,4.0,430,408.0,-22.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,2.0,-8.0,12.0,14.0,280.0,279.0,263.0,2330,737.0,4.0,750,741.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,18.0,-2.0,16.0,34.0,286.0,293.0,266.0,2296,800.0,11.0,806,811.0,5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,15.0,-5.0,15.0,30.0,285.0,281.0,258.0,2342,748.0,8.0,805,756.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,24.0,-1.0,11.0,35.0,235.0,215.0,199.0,1448,254.0,5.0,320,259.0,-21.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


---
### Task 1: Select Data

Decide on the data to be used for analysis. Consider which columns (features) and rows (records) to include or exclude based on:

- **Relevance:** Does this feature contribute to the data mining goal?
- **Data Quality:** Is the quality of this feature sufficient (e.g., too many missing values)?
- **Technical Constraints:** Are there limitations on data volume or specific feature types?

**Output:** A rationale for inclusion/exclusion of data, and the resulting subset.

**Instructions:** Select the columns and rows relevant to your analysis goal. Document your reasoning.

In [3]:
# TODO: Select the relevant columns and rows for your analysis.
# Document your rationale for including or excluding specific features.

df_selected, columns_to_keep, columns_to_drop = select_phase3_data(df)
drop_reason = {
    "post_departure_and_arrival_fields": "Dropped because they would leak outcome information that would not be available at prediction time.",
    "cancellation_reason": "Dropped because it already describes the outcome.",
    "unused_identifiers": "Dropped because they were outside the main Q1 classification scope."
}

print("Columns kept for the main Q1 modelling scope:")
print(columns_to_keep)

print("\nColumns excluded from the modelling dataset:")
print(columns_to_drop)

print("\nReasons for exclusion:")
for key, value in drop_reason.items():
    print(f"- {key}: {value}")

print(f"\nShape after column selection: {df_selected.shape}")

Columns kept for the main Q1 modelling scope:
['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'SCHEDULED_TIME', 'DISTANCE', 'DEPARTURE_DELAY', 'CANCELLED', 'DIVERTED']

Columns excluded from the modelling dataset:
['DEPARTURE_TIME', 'TAXI_OUT', 'WHEELS_OFF', 'ELAPSED_TIME', 'AIR_TIME', 'WHEELS_ON', 'TAXI_IN', 'ARRIVAL_TIME', 'ARRIVAL_DELAY', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY', 'CANCELLATION_REASON']

Reasons for exclusion:
- post_departure_and_arrival_fields: Dropped because they would leak outcome information that would not be available at prediction time.
- cancellation_reason: Dropped because it already describes the outcome.
- unused_identifiers: Dropped because they were outside the main Q1 classification scope.

Shape after column selection: (5819079, 14)


In [4]:
# Optional: Filter rows based on specific criteria
# Example: Remove rows where a critical field is missing or filter by a condition

df_selected = filter_phase3_rows(df_selected)
print(f"Shape after row selection: {df_selected.shape}")
df_selected.head()

Shape after row selection: (5714008, 14)


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,SCHEDULED_TIME,DISTANCE,DEPARTURE_DELAY,CANCELLED,DIVERTED
0,2015,1,1,4,AS,98,ANC,SEA,5,205.0,1448,-11.0,0,0
1,2015,1,1,4,AA,2336,LAX,PBI,10,280.0,2330,-8.0,0,0
2,2015,1,1,4,US,840,SFO,CLT,20,286.0,2296,-2.0,0,0
3,2015,1,1,4,AA,258,LAX,MIA,20,285.0,2342,-5.0,0,0
4,2015,1,1,4,AS,135,SEA,ANC,25,235.0,1448,-1.0,0,0


---
### Task 2: Clean Data

Raise data quality to the level required by the selected analysis techniques. Cleaning activities include:

- **Handle Missing Values:** Impute missing values (mean, median, mode, forward/backward fill) or remove rows/columns with excessive missing data.
- **Correct Errors:** Fix inaccurate or corrupted data entries.
- **Remove Duplicates:** Eliminate exact or near-duplicate records.
- **Handle Outliers:** Decide how to treat extreme values (keep, cap, transform, or remove).

**Instructions:** Apply appropriate cleaning techniques to address the data quality issues identified in Phase 2, Task 4.

In [5]:
# TODO: Handle missing values.
# Choose an appropriate strategy for each column.

df_clean = clean_phase3_missing_values(df_selected)
print(f"Missing values remaining: {df_clean.isnull().sum().sum()}")

Missing values remaining: 0


In [6]:
# TODO: Remove duplicate records.

df_clean, removed_duplicates = drop_duplicate_rows(df_clean)
print(f"Removed {removed_duplicates} duplicate rows. Remaining: {len(df_clean)} rows.")

Removed 0 duplicate rows. Remaining: 5714008 rows.


In [7]:
# TODO: Handle outliers.
# Choose a strategy: capping (winsorizing), removing, or transforming.

outlier_columns = ['DISTANCE', 'SCHEDULED_TIME']
df_clean, outlier_summary = cap_outliers_iqr(df_clean, outlier_columns)
print("Outlier treatment applied using IQR capping to:", outlier_columns)
outlier_summary

Outlier treatment applied using IQR capping to: ['DISTANCE', 'SCHEDULED_TIME']


,lower_bound,upper_bound,capped_values
DISTANCE,0.0,2103.0,345365.0
SCHEDULED_TIME,0.0,307.5,289411.0


---
### Task 3: Construct Data (Feature Engineering)

This task involves creating new attributes (features) derived from existing ones that may be more useful for modelling. Common techniques include:

- **Derived Attributes:** Create new features from existing ones (e.g., extracting `year`, `month`, `day` from a datetime column; computing `total_spend = price * quantity`).
- **Binning / Discretisation:** Convert continuous variables into categorical bins (e.g., age groups).
- **Encoding Categorical Variables:** Convert categorical features into numerical representations (e.g., one-hot encoding, label encoding).
- **Scaling / Normalisation:** Scale numerical features to a common range (e.g., Min-Max scaling, Standardisation).

**Instructions:** Create new features or transform existing ones to improve model performance.

In [19]:
# TODO: Create derived attributes / new features.

df_clean = add_phase3_features(df_clean)
df_clean = df_clean.drop(columns=["DEPARTURE_DELAY"])
print("Created features:")
print([TARGET_COLUMN, 'SCHEDULED_DEPARTURE_HOUR'])
df_clean[[TARGET_COLUMN, 'SCHEDULED_DEPARTURE_HOUR']].head()

Created features:
['DELAY_15', 'SCHEDULED_DEPARTURE_HOUR']


,DELAY_15,SCHEDULED_DEPARTURE_HOUR
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


In [20]:
# TODO: Encode categorical variables.

categorical_columns = df_clean.select_dtypes(include='object').columns.tolist()
print("Categorical columns identified for later encoding in Phase 4:")
print(categorical_columns)
print("\nEncoding is intentionally deferred to the sklearn pipeline in Phase 4 to keep preprocessing reproducible and avoid unnecessary expansion at this stage.")

Categorical columns identified for later encoding in Phase 4:
['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']

Encoding is intentionally deferred to the sklearn pipeline in Phase 4 to keep preprocessing reproducible and avoid unnecessary expansion at this stage.


In [21]:
# TODO: Scale / normalise numerical features if required.

numerical_columns = df_clean.select_dtypes(include=np.number).columns.tolist()
print("Numerical columns available for later scaling in Phase 4:")
print(numerical_columns)
print("\nScaling is not applied in Phase 3 because tree-based models do not require it, and any scaling needed for linear models can be handled more safely inside the Phase 4 sklearn pipeline.")

Numerical columns available for later scaling in Phase 4:
['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'FLIGHT_NUMBER', 'SCHEDULED_DEPARTURE', 'SCHEDULED_TIME', 'DISTANCE', 'CANCELLED', 'DIVERTED', 'DELAY_15', 'SCHEDULED_DEPARTURE_HOUR']

Scaling is not applied in Phase 3 because tree-based models do not require it, and any scaling needed for linear models can be handled more safely inside the Phase 4 sklearn pipeline.


---
### Task 4: Integrate Data

If your project uses multiple data sources, this task involves merging or combining them into a single, unified dataset. Activities include:

- **Merging Tables:** Join datasets on common keys (e.g., using `pd.merge()`).
- **Appending Records:** Concatenate datasets with the same structure (e.g., using `pd.concat()`).
- **Aggregation:** Summarise data at a different level of granularity.

**Instructions:** If using multiple data sources, merge or concatenate them below. If your project uses a single dataset, document that here and proceed to the next task.

In [22]:
# TODO: Integrate data from multiple sources (if applicable).

airlines_df = load_airlines(AIRLINES_PATH)
airports_df = load_airports(AIRPORTS_PATH)

df_integrated = merge_airline_lookup(df_clean, airlines_df)
df_integrated = merge_airport_lookup(df_integrated, airports_df)
print(f"Integrated dataset shape: {df_integrated.shape}")

Integrated dataset shape: (5714008, 28)


In [23]:
# Optional: Verify the integrated data
df_integrated.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,SCHEDULED_TIME,DISTANCE,CANCELLED,DIVERTED,DELAY_15,SCHEDULED_DEPARTURE_HOUR,AIRLINE_NAME,ORIGIN_AIRPORT_NAME,ORIGIN_CITY,ORIGIN_STATE,ORIGIN_COUNTRY,ORIGIN_LATITUDE,ORIGIN_LONGITUDE,DESTINATION_AIRPORT_NAME,DESTINATION_CITY,DESTINATION_STATE,DESTINATION_COUNTRY,DESTINATION_LATITUDE,DESTINATION_LONGITUDE
0,2015,1,1,4,AS,98,ANC,SEA,5,205.0,1448,0,0,0,0,Alaska Airlines Inc.,Ted Stevens Anchorage International Airport,Anchorage,AK,USA,61.17432,-149.99619,Seattle-Tacoma International Airport,Seattle,WA,USA,47.44898,-122.30931
1,2015,1,1,4,AA,2336,LAX,PBI,10,280.0,2103,0,0,0,0,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807,Palm Beach International Airport,West Palm Beach,FL,USA,26.68316,-80.09559
2,2015,1,1,4,US,840,SFO,CLT,20,286.0,2103,0,0,0,0,US Airways Inc.,San Francisco International Airport,San Francisco,CA,USA,37.61900,-122.37484,Charlotte Douglas International Airport,Charlotte,NC,USA,35.21401,-80.94313
3,2015,1,1,4,AA,258,LAX,MIA,20,285.0,2103,0,0,0,0,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807,Miami International Airport,Miami,FL,USA,25.79325,-80.29056
4,2015,1,1,4,AS,135,SEA,ANC,25,235.0,1448,0,0,0,0,Alaska Airlines Inc.,Seattle-Tacoma International Airport,Seattle,WA,USA,47.44898,-122.30931,Ted Stevens Anchorage International Airport,Anchorage,AK,USA,61.17432,-149.99619


---
### Task 5: Format Data

This final preparation task ensures the data is in the correct format for the modelling tools. Activities include:

- **Data Type Conversions:** Ensure all columns have appropriate data types (e.g., numeric, datetime, categorical).
- **Column Reordering:** Arrange columns in a logical order (e.g., features first, target last).
- **Renaming:** Give columns clear, descriptive names.
- **Saving the Prepared Dataset:** Export the final, clean dataset for use in subsequent phases.

**Instructions:** Apply any final formatting changes and save the prepared dataset.

In [24]:
# TODO: Apply final formatting ? data types, column order, renaming.

df_integrated['FLIGHT_NUMBER'] = df_integrated['FLIGHT_NUMBER'].astype(str)
df_integrated[TARGET_COLUMN] = df_integrated[TARGET_COLUMN].astype(int)
df_integrated['SCHEDULED_DEPARTURE_HOUR'] = df_integrated['SCHEDULED_DEPARTURE_HOUR'].astype(int)

df_integrated = df_integrated.rename(columns={
    'AIRLINE': 'AIRLINE_CODE'
})

target_col = TARGET_COLUMN
feature_cols = [col for col in df_integrated.columns if col != target_col]
df_final = df_integrated[feature_cols + [target_col]]
df_final.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE_CODE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,SCHEDULED_TIME,DISTANCE,CANCELLED,DIVERTED,SCHEDULED_DEPARTURE_HOUR,AIRLINE_NAME,ORIGIN_AIRPORT_NAME,ORIGIN_CITY,ORIGIN_STATE,ORIGIN_COUNTRY,ORIGIN_LATITUDE,ORIGIN_LONGITUDE,DESTINATION_AIRPORT_NAME,DESTINATION_CITY,DESTINATION_STATE,DESTINATION_COUNTRY,DESTINATION_LATITUDE,DESTINATION_LONGITUDE,DELAY_15
0,2015,1,1,4,AS,98,ANC,SEA,5,205.0,1448,0,0,0,Alaska Airlines Inc.,Ted Stevens Anchorage International Airport,Anchorage,AK,USA,61.17432,-149.99619,Seattle-Tacoma International Airport,Seattle,WA,USA,47.44898,-122.30931,0
1,2015,1,1,4,AA,2336,LAX,PBI,10,280.0,2103,0,0,0,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807,Palm Beach International Airport,West Palm Beach,FL,USA,26.68316,-80.09559,0
2,2015,1,1,4,US,840,SFO,CLT,20,286.0,2103,0,0,0,US Airways Inc.,San Francisco International Airport,San Francisco,CA,USA,37.61900,-122.37484,Charlotte Douglas International Airport,Charlotte,NC,USA,35.21401,-80.94313,0
3,2015,1,1,4,AA,258,LAX,MIA,20,285.0,2103,0,0,0,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807,Miami International Airport,Miami,FL,USA,25.79325,-80.29056,0
4,2015,1,1,4,AS,135,SEA,ANC,25,235.0,1448,0,0,0,Alaska Airlines Inc.,Seattle-Tacoma International Airport,Seattle,WA,USA,47.44898,-122.30931,Ted Stevens Anchorage International Airport,Anchorage,AK,USA,61.17432,-149.99619,0


In [25]:
# TODO: Verify the final prepared dataset.

print("=" * 60)
print("FINAL PREPARED DATASET SUMMARY")
print("=" * 60)
print(f"Shape: {df_final.shape}")
print(f"Missing values: {df_final.isnull().sum().sum()}")
print(f"Duplicates: {df_final.duplicated().sum()}")
print(f"\nColumn types:")
print(df_final.dtypes)
print(f"\nTarget balance:")
print(df_final[TARGET_COLUMN].value_counts(normalize=True))
df_final.head()

FINAL PREPARED DATASET SUMMARY
Shape: (5714008, 28)
Missing values: 5812796
Duplicates: 0

Column types:
YEAR                          int64
MONTH                         int64
DAY                           int64
DAY_OF_WEEK                   int64
AIRLINE_CODE                 object
FLIGHT_NUMBER                object
ORIGIN_AIRPORT               object
DESTINATION_AIRPORT          object
SCHEDULED_DEPARTURE           int64
SCHEDULED_TIME              float64
DISTANCE                      int64
CANCELLED                     int64
DIVERTED                      int64
SCHEDULED_DEPARTURE_HOUR      int64
AIRLINE_NAME                 object
ORIGIN_AIRPORT_NAME          object
ORIGIN_CITY                  object
ORIGIN_STATE                 object
ORIGIN_COUNTRY               object
ORIGIN_LATITUDE             float64
ORIGIN_LONGITUDE            float64
DESTINATION_AIRPORT_NAME     object
DESTINATION_CITY             object
DESTINATION_STATE            object
DESTINATION_COUNTRY          ob

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE_CODE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,SCHEDULED_TIME,DISTANCE,CANCELLED,DIVERTED,SCHEDULED_DEPARTURE_HOUR,AIRLINE_NAME,ORIGIN_AIRPORT_NAME,ORIGIN_CITY,ORIGIN_STATE,ORIGIN_COUNTRY,ORIGIN_LATITUDE,ORIGIN_LONGITUDE,DESTINATION_AIRPORT_NAME,DESTINATION_CITY,DESTINATION_STATE,DESTINATION_COUNTRY,DESTINATION_LATITUDE,DESTINATION_LONGITUDE,DELAY_15
0,2015,1,1,4,AS,98,ANC,SEA,5,205.0,1448,0,0,0,Alaska Airlines Inc.,Ted Stevens Anchorage International Airport,Anchorage,AK,USA,61.17432,-149.99619,Seattle-Tacoma International Airport,Seattle,WA,USA,47.44898,-122.30931,0
1,2015,1,1,4,AA,2336,LAX,PBI,10,280.0,2103,0,0,0,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807,Palm Beach International Airport,West Palm Beach,FL,USA,26.68316,-80.09559,0
2,2015,1,1,4,US,840,SFO,CLT,20,286.0,2103,0,0,0,US Airways Inc.,San Francisco International Airport,San Francisco,CA,USA,37.61900,-122.37484,Charlotte Douglas International Airport,Charlotte,NC,USA,35.21401,-80.94313,0
3,2015,1,1,4,AA,258,LAX,MIA,20,285.0,2103,0,0,0,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807,Miami International Airport,Miami,FL,USA,25.79325,-80.29056,0
4,2015,1,1,4,AS,135,SEA,ANC,25,235.0,1448,0,0,0,Alaska Airlines Inc.,Seattle-Tacoma International Airport,Seattle,WA,USA,47.44898,-122.30931,Ted Stevens Anchorage International Airport,Anchorage,AK,USA,61.17432,-149.99619,0


In [26]:
# TODO: Save the prepared dataset for use in Phase 4 (Modelling).

OUTPUT_PATH = PREPARED_DATA_PATH
saved_path = save_prepared_dataset(df_final, OUTPUT_PATH)
print(f"Prepared dataset saved to: {saved_path}")

Prepared dataset saved to: C:\Users\anasa\KH5004_Portfolio\data\processed\flights_prepared_phase3.csv
